# Estimación de ancestría genética global a partir de un archivo VCF

En este cuaderno vamos a estimar la **ancestría genética global**: qué proporción del genoma de cada persona proviene de cada población ancestral. Trabajaremos con datos reales de exoma de 969 personas de 15 poblaciones.

Pasos:

1. Descargar PLINK 2
2. Descargar los datos
3. Convertir a formato binario de PLINK
4. Control de calidad
5. Análisis de componentes principales (PCA)
6. Descargar ADMIXTURE
7. Correr ADMIXTURE
8. Graficar los resultados de ADMIXTURE

Después de cada paso hay una tabla con los archivos que crea, qué son y si se usan más adelante. El resumen completo, con un diagrama de cómo se conectan, está en [archivos_por_paso.md](https://github.com/YTZ27/curso-ancestria/blob/main/notebooks/archivos_por_paso.md).

## 1. Descargar PLINK 2

Si el enlace deja de funcionar, entra a la [página de PLINK 2](https://www.cog-genomics.org/plink/2.0/), copia el enlace de **Linux 64-bit Intel** de la columna **Alpha** y pégalo en la línea del `wget`. El resto de la celda no cambia.

In [ ]:
!wget -nv -O plink2.zip https://s3.amazonaws.com/plink2-assets/alpha7/plink2_linux_x86_64_20261015.zip
!unzip -o plink2.zip
!./plink2 --version
!ls -lh plink2.zip plink2

**Archivos de este paso**

| Archivo | Qué es | ¿Se usa después? |
|---|---|---|
| `plink2.zip` | La descarga comprimida | No: solo sirve para extraer `plink2` |
| `plink2` | El programa PLINK 2 | Sí: en los pasos 3, 4 y 5 |

## 2. Descargar los datos

El archivo `exome.vcf.gz` está en el repositorio del curso (carpeta `data/`). Contiene personas del Proyecto 1000 Genomas y del Human Genome Diversity Panel (HGDP), restringidas a las regiones codificantes: los archivos VCF originales ([recursos de PLINK 2](https://www.cog-genomics.org/plink/2.0/resources)) se filtraron con el archivo BED del kit de exoma [Agilent SureSelectXT Human All Exon V8](https://www.agilent.com/en/product/next-generation-sequencing/ngs-assays-panels/exome/sureselect-human-all-exon-4252439).

In [ ]:
!wget -nv -nc https://github.com/YTZ27/curso-ancestria/raw/main/data/exome.vcf.gz
!ls -lh exome.vcf.gz

**Archivos de este paso**

| Archivo | Qué es | ¿Se usa después? |
|---|---|---|
| `exome.vcf.gz` | Genotipos de 969 personas en 244 305 variantes, en formato VCF (una tabla de texto) comprimido con gzip (`.gz`) | Sí: es la entrada del paso 3 |

Veamos qué hay por dentro. Las líneas que empiezan con `##` son el encabezado; después viene una tabla con una fila por variante. Abajo salen las 3 primeras variantes y las 3 primeras personas.

Columnas: `#CHROM` (cromosoma), `POS` (posición), `ID`, `REF` (alelo de referencia), `ALT` (alelo alternativo), `QUAL`, `FILTER`, `INFO`, `FORMAT`, y luego una columna por persona: `0/0` = dos alelos de referencia, `0/1` = uno de cada, `1/1` = dos alelos alternativos, `./.` = dato faltante.

In [ ]:
!zcat exome.vcf.gz | grep -v '^##' | head -4 | cut -f1-12

## 3. Convertir al formato binario de PLINK

PLINK guarda cada genotipo en solo 2 bits, así que sus archivos binarios se procesan rápido, y ADMIXTURE los lee directamente. La conversión crea tres archivos que siempre van juntos: `.bed` (genotipos), `.bim` (variantes) y `.fam` (personas).

In [ ]:
!./plink2 --vcf exome.vcf.gz --make-bed --out exome_data
!ls -lh exome_data.*

**Archivos de este paso**

| Archivo | Qué es | ¿Se usa después? |
|---|---|---|
| `exome_data.bed` | Genotipos en formato binario. No se puede leer como texto: no lo abras | Sí: entrada del paso 4 |
| `exome_data.bim` | Una fila por variante: cromosoma, identificador, posición genética (0 = desconocida), posición en pares de bases, alelo 1, alelo 2 | Sí: entrada del paso 4 |
| `exome_data.fam` | Una fila por persona: identificador de familia, identificador de la persona, padre, madre, sexo (0 = desconocido), fenotipo (-9 = faltante) | Sí: entrada del paso 4 |
| `exome_data.log` | Registro del comando y resumen del resultado | No, pero es donde se revisa qué hizo PLINK |

PLINK lee los tres archivos por su nombre común (prefijo): `--bfile exome_data` significa `exome_data.bed` + `exome_data.bim` + `exome_data.fam`.

Las primeras filas del `.bim` (variantes) y del `.fam` (personas):

In [ ]:
!head -3 exome_data.bim
!head -3 exome_data.fam

## 4. Control de calidad y poda por LD

Aplicamos un control de calidad estándar que conserva las variantes:

* con frecuencia de al menos 1 % (`--maf 0.01`)
* genotipadas en al menos el 90 % de las personas (`--geno 0.1`)

Y las personas:

* con al menos el 90 % de sus variantes sin datos faltantes (`--mind 0.1`)

In [ ]:
!./plink2 --bfile exome_data \
       --maf 0.01 \
       --geno 0.1 \
       --mind 0.1 \
       --make-bed \
       --out qc_maf-0.01_geno-0.1
!ls -lh qc_maf-0.01_geno-0.1.*
!grep -E "removed|remaining" qc_maf-0.01_geno-0.1.log

**Archivos de este paso**

| Archivo | Qué es | ¿Se usa después? |
|---|---|---|
| `qc_maf-0.01_geno-0.1.bed / .bim / .fam` | Los mismos datos, conservando solo las variantes y personas que pasaron los filtros. El nombre indica los filtros usados | Sí: entrada de la poda por LD (pasos 4.1 y 4.2) |
| `qc_maf-0.01_geno-0.1.log` | Resumen de los filtros | No. La última línea de la celda muestra sus líneas clave: cuántas variantes y personas eliminó cada filtro |

Con los datos del curso no se elimina ninguna persona, 133 675 variantes tienen frecuencia menor al 1 % y quedan 110 630 variantes.

### ¿Por qué no filtramos por equilibrio de Hardy-Weinberg?

En un control de calidad típico también se eliminan las variantes que se desvían del **equilibrio de Hardy-Weinberg** (`--hwe`), porque una desviación fuerte suele indicar un error de genotipado (Anderson et al., 2010).

Aquí no lo hacemos, y la razón es el objetivo del análisis. El equilibrio de Hardy-Weinberg supone **una sola población que se cruza al azar**. Nuestros datos son 15 poblaciones juntas, y cuando se mezclan poblaciones con frecuencias alélicas distintas, el conjunto muestra menos heterocigotos de los esperados aunque cada población por separado esté en equilibrio. Es el **efecto Wahlund** (Hartl y Clark, 2007).

El problema es que ese efecto es más fuerte justamente en las variantes que más difieren entre continentes, que son las informativas de ancestría. Con estos datos, el filtro `--hwe 0.0001` elimina el **81,5 %** de las variantes cuya frecuencia difiere en más de 0,5 entre continentes. En las variantes que eliminaba, la mediana de esa diferencia era 0,45; en las que conservaba, 0,08.

Dicho de otro modo: el filtro estaba quitando la señal que queremos medir.

**Qué hacer en un análisis propio:** si quieres filtrar por Hardy-Weinberg, aplícalo **dentro de cada población por separado** (con `--keep`) y elimina solo las variantes que fallan en varias poblaciones a la vez. Si el objetivo es estimar ancestría, omitirlo es una opción razonable.

### Poda por desequilibrio de ligamiento (LD)

Se hace en dos pasos:

* Ventanas de 50 variantes, avanzando de 5 en 5 (`50 5`)
* En cada ventana eliminamos una variante de cada par con r² > 0,2

**Paso 4.1:** generar la lista de variantes a conservar.

In [ ]:
!./plink2 --bfile qc_maf-0.01_geno-0.1 \
       --indep-pairwise 50 5 0.2 \
       --out ld_pruned
!wc -l ld_pruned.prune.in ld_pruned.prune.out

**Archivos de este paso**

| Archivo | Qué es | ¿Se usa después? |
|---|---|---|
| `ld_pruned.prune.in` | Lista de identificadores de las variantes que se conservan (aproximadamente independientes entre sí) | Sí: en el paso 4.2 |
| `ld_pruned.prune.out` | Lista de identificadores eliminados por estar en LD con una variante que se conserva | No: solo para revisar |
| `ld_pruned.log` | Registro del comando | No |

Este paso no modifica los datos: solo escribe listas. Con los datos del curso se conservan 59 850 variantes y se eliminan 50 780.

**Paso 4.2:** conservar solo las variantes de `ld_pruned.prune.in`.

In [ ]:
!./plink2 --bfile qc_maf-0.01_geno-0.1 \
       --extract ld_pruned.prune.in \
       --make-bed \
       --out admixture_LD_QC
!ls -lh admixture_LD_QC.*

**Archivos de este paso**

| Archivo | Qué es | ¿Se usa después? |
|---|---|---|
| `admixture_LD_QC.bed / .bim / .fam` | Datos finales: pasaron el control de calidad y la poda por LD (969 personas, 59 850 variantes) | Sí: entrada del PCA (paso 5) y de ADMIXTURE (paso 7). El `.fam` además identifica a cada persona en los gráficos |
| `admixture_LD_QC.log` | Registro del comando | No |

**¿Por qué podar por LD?** El PCA y ADMIXTURE suponen que las variantes son independientes. Las variantes en LD repiten la misma información y le darían demasiado peso a algunas regiones del genoma (Alexander et al., 2009).

## 5. Análisis de componentes principales (PCA)

El PCA resume las diferencias genéticas entre todas las personas en unos pocos ejes nuevos, los componentes principales (PCs). Las personas con ancestría parecida quedan cerca unas de otras. A diferencia de ADMIXTURE, el PCA no necesita que elijamos un número de poblaciones ancestrales.

Usamos los datos finales del paso 4 y calculamos los primeros 10 PCs.

In [ ]:
!./plink2 --bfile admixture_LD_QC --pca 10 --out pca
!ls -lh pca.*

**Archivos de este paso**

| Archivo | Qué es | ¿Se usa después? |
|---|---|---|
| `pca.eigenvec` | Una fila por persona (`#FID`, `IID`) con su posición en PC1 a PC10 | Sí: para el gráfico de PC1 vs PC2 |
| `pca.eigenval` | 10 números, uno por PC: cuánta variación captura cada uno | Sí: para el gráfico de eigenvalores |
| `pca.log` | Registro del comando | No |

In [ ]:
!head -3 pca.eigenvec
!cat pca.eigenval

¿Cuántos PCs importan? Los primeros capturan casi toda la estructura poblacional; después, los valores se aplanan.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

eigenval = pd.read_csv('pca.eigenval', header=None)[0]
ax = eigenval.plot.bar(color='#6b6b6b', figsize=(6, 3), width=0.7)
ax.set_xticklabels(range(1, len(eigenval) + 1), rotation=0)
ax.set(xlabel='Componente principal', ylabel='Eigenvalor', title='Variación que captura cada PC')
for lado in ['top', 'right']:
    ax.spines[lado].set_visible(False)
plt.tight_layout()
plt.show()

Ahora leemos los PCs y le agregamos a cada persona su población y, en el caso de las poblaciones de referencia, su continente.

In [ ]:
pcs = pd.read_csv('pca.eigenvec', sep=r'\s+')
pcs['poblacion'] = pcs['IID'].str.split('_').str[0]

continente = {'forReferenceYRI': 'Africana', 'forReferenceESN': 'Africana', 'forReferenceGWD': 'Africana',
              'forReferenceGBR': 'Europea', 'forReferenceIBS': 'Europea', 'forReferenceTSI': 'Europea',
              'forReferenceKaritiana': 'Nativa americana', 'forReferenceSurui': 'Nativa americana',
              'forReferencePima': 'Nativa americana', 'forReferenceMaya': 'Nativa americana',
              'forReferencePEL': 'Nativa americana'}
pcs['continente'] = pcs['poblacion'].map(continente)  # vacío (NaN) en las poblaciones mezcladas
pcs.head()

PC1 contra PC2: las poblaciones de referencia van coloreadas por continente, y cada panel muestra una población mezclada en negro.

In [ ]:
colores = {'Africana': '#eb6834', 'Europea': '#2a78d6', 'Nativa americana': '#1baf7a'}
fig, axes = plt.subplots(2, 2, figsize=(10, 9), sharex=True, sharey=True)

for ax, pob in zip(axes.flat, ['ACB', 'ASW', 'PUR', 'MXL']):
    for nombre, color in colores.items():
        ref = pcs[pcs['continente'] == nombre]
        ax.scatter(ref['PC1'], ref['PC2'], s=14, color=color, alpha=0.5, label=f'Referencia {nombre.lower()}')
    mez = pcs[pcs['poblacion'] == pob]
    ax.scatter(mez['PC1'], mez['PC2'], s=14, color='black', label='Población mezclada')
    ax.set_title(pob)

axes[0, 0].legend(frameon=False, fontsize=9)
for ax in axes[1]:
    ax.set_xlabel('PC1')
for ax in axes[:, 0]:
    ax.set_ylabel('PC2')
plt.tight_layout()
plt.show()

**Cómo leerlo:** cada continente forma su propio grupo, y las personas mezcladas caen *entre* los grupos de sus poblaciones de origen; mientras más cerca está una persona de un grupo, mayor es esa ancestría. ADMIXTURE le pondrá números a esto en los pasos siguientes.

El signo de un PC es arbitrario: tu gráfico puede salir volteado respecto al de otra persona, y significa lo mismo.

Los gráficos de este paso se muestran aquí y no se guardan como archivos.

## 6. Descargar ADMIXTURE

In [ ]:
!wget -nv -nc https://dalexander.github.io/admixture/binaries/admixture_linux-1.3.0.tar.gz
!tar -xzf admixture_linux-1.3.0.tar.gz
!ls -lh dist/admixture_linux-1.3.0/

**Archivos de este paso**

| Archivo | Qué es | ¿Se usa después? |
|---|---|---|
| `admixture_linux-1.3.0.tar.gz` | La descarga comprimida | No: solo sirve para extraer la carpeta `dist/` |
| `dist/admixture_linux-1.3.0/admixture` | El programa ADMIXTURE | Sí: en el paso 7 |
| `dist/admixture_linux-1.3.0/admixture32` | Versión para computadores antiguos de 32 bits | No |
| `dist/admixture_linux-1.3.0/admixture-manual.pdf` | El manual de ADMIXTURE | No, pero vale la pena leerlo |
| `dist/admixture_linux-1.3.0/README.32.txt` | Notas sobre la versión de 32 bits | No |

## 7. Correr ADMIXTURE

Corremos ADMIXTURE en modo no supervisado con K = 3, el número de poblaciones ancestrales. `-j2` usa los 2 procesadores que da Colab. Tarda varios minutos.

In [ ]:
!./dist/admixture_linux-1.3.0/admixture -j2 admixture_LD_QC.bed 3
!ls -lh admixture_LD_QC.3.*

**Archivos de este paso**

| Archivo | Qué es | ¿Se usa después? |
|---|---|---|
| `admixture_LD_QC.3.Q` | Proporciones de ancestría: una fila por persona (mismo orden que el `.fam`) y una columna por componente ancestral; cada fila suma 1 | Sí: para el gráfico del paso 8, junto con `admixture_LD_QC.fam` para saber de quién es cada fila |
| `admixture_LD_QC.3.P` | Frecuencia alélica de cada variante en cada componente ancestral: una fila por variante (mismo orden que el `.bim`) y una columna por componente | No en este cuaderno. Sirve para proyectar personas nuevas sobre los mismos componentes |

El `3` de los nombres es K. ADMIXTURE no escribe archivo `.log`: sus mensajes solo aparecen en la salida de la celda.

In [ ]:
!head admixture_LD_QC.3.Q

In [ ]:
!head admixture_LD_QC.fam

## 8. Graficar los resultados de ADMIXTURE

Primero leemos los resultados y la población de cada persona (la primera parte de su identificador).

In [ ]:
Q = pd.read_csv('admixture_LD_QC.3.Q', sep=r'\s+', header=None)
fam = pd.read_csv('admixture_LD_QC.fam', sep=r'\s+', header=None)
Q['poblacion'] = fam[1].str.split('_').str[0]
Q.head()

ADMIXTURE no sabe cuál componente es africano, europeo o nativo americano: solo los numera (0, 1, 2). Le ponemos a cada componente el nombre de la población de referencia donde es más alto.

In [ ]:
refs = {'forReferenceYRI': 'Africana', 'forReferenceGBR': 'Europea', 'forReferenceKaritiana': 'Nativa americana'}
promedios = Q.groupby('poblacion')[[0, 1, 2]].mean().loc[list(refs)]
promedios.round(2)

In [ ]:
nombres = {k: refs[promedios[k].idxmax()] for k in [0, 1, 2]}
Q = Q.rename(columns=nombres)
nombres

Ahora ordenamos a las personas: primero las poblaciones de referencia (África, Europa, América) y después las mezcladas.

In [ ]:
orden = ['forReferenceYRI', 'forReferenceESN', 'forReferenceGWD',
         'forReferenceGBR', 'forReferenceIBS', 'forReferenceTSI',
         'forReferenceKaritiana', 'forReferenceSurui', 'forReferencePima', 'forReferenceMaya', 'forReferencePEL',
         'ACB', 'ASW', 'PUR', 'MXL']
Q['poblacion'] = pd.Categorical(Q['poblacion'], categories=orden, ordered=True)
Q = Q.sort_values(['poblacion', 'Europea'], ascending=[True, False]).reset_index(drop=True)

Y el gráfico: una barra por persona, agrupadas por población.

In [ ]:
ax = Q[list(colores)].plot.bar(stacked=True, width=1, figsize=(20, 5), color=colores, edgecolor='none')

# Una etiqueta por población, en la mitad de su bloque, y una línea blanca entre bloques
tam = Q.groupby('poblacion', observed=True).size()
fin = tam.cumsum()
ax.set_xticks((fin - tam / 2 - 0.5).values)
ax.set_xticklabels(tam.index.astype(str).str.replace('forReference', 'ref: '), rotation=90)
for x in fin.iloc[:-1]:
    ax.axvline(x - 0.5, color='white', linewidth=1)

ax.set(ylim=(0, 1), xlabel='', ylabel='Proporción de ancestría', title='ADMIXTURE, K = 3')
ax.legend(title='Ancestría', bbox_to_anchor=(1, 1), frameon=False)
for lado in ax.spines.values():
    lado.set_visible(False)
plt.tight_layout()
plt.show()

**Cómo leerlo:** cada barra es una persona y cada color, la proporción de su genoma que viene de ese componente ancestral. Las poblaciones de referencia salen casi de un solo color, y las mezcladas combinan varios.

**Cuidado al interpretar:** los componentes de ADMIXTURE no son poblaciones ancestrales reales, sino grupos estadísticos que dependen de K y de qué poblaciones incluimos en el análisis. Un gráfico con otro K o con otras poblaciones puede contar una historia distinta (Lawson et al., 2018).

Este paso no crea archivos: el gráfico se muestra aquí. Para guardarlo, clic derecho sobre la imagen, o agregar `plt.savefig('admixture_K3.png', dpi=300)` antes de `plt.show()`.

## Referencias

- Alexander DH, Novembre J, Lange K. Fast model-based estimation of ancestry in unrelated individuals. *Genome Res*. 2009;19:1655–1664. doi:10.1101/gr.094052.109
- Anderson CA, et al. Data quality control in genetic case-control association studies. *Nat Protoc*. 2010;5:1564–1573. doi:10.1038/nprot.2010.116
- Chang CC, et al. Second-generation PLINK: rising to the challenge of larger and richer datasets. *GigaScience*. 2015;4:7. doi:10.1186/s13742-015-0047-8
- Hartl DL, Clark AG. *Principles of Population Genetics*. 4.ª ed. Sinauer; 2007 (efecto Wahlund)
- Lawson DJ, van Dorp L, Falush D. A tutorial on how not to over-interpret STRUCTURE and ADMIXTURE bar plots. *Nat Commun*. 2018;9:3258. doi:10.1038/s41467-018-05257-7
- Patterson N, Price AL, Reich D. Population structure and eigenanalysis. *PLoS Genet*. 2006;2(12):e190. doi:10.1371/journal.pgen.0020190
- 1000 Genomes Project Consortium. A global reference for human genetic variation. *Nature*. 2015;526:68–74. doi:10.1038/nature15393